In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets
from torch.utils.data import DataLoader
from transformers import ViTImageProcessor, ViTForImageClassification
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score
from collections import Counter
import os

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [14]:
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

def transform(image):
    image = image.convert("RGB")
    inputs = processor(images=image, return_tensors="pt")
    return inputs["pixel_values"].squeeze(0)

In [15]:
train_dataset = datasets.ImageFolder("Dataset/Train", transform=transform)
val_dataset = datasets.ImageFolder("Dataset/Validation", transform=transform)
test_dataset = datasets.ImageFolder("Dataset/Test", transform=transform)

print("Class mapping:", train_dataset.class_to_idx)

Class mapping: {'Fake': 0, 'Real': 1}


In [16]:
BATCH_SIZE = 16  # safe value for RTX 4060 laptop

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers = True
)

In [21]:
checkpoint_path = "checkpoints/best_model.pth"

model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=2,
    ignore_mismatched_sizes=True
)

model.to(device)

optimizer = optim.AdamW(model.parameters(), lr=5e-5)

start_epoch = 0
best_val_acc = 0

if os.path.exists(checkpoint_path):
    print("Checkpoint found. Loading...")

    checkpoint = torch.load(checkpoint_path, map_location=device)

    # ---- LOAD MODEL ----
    model.load_state_dict(checkpoint['model_state_dict'])

    # ---- LOAD OPTIMIZER ----
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    # ---- LOAD SCALER (AMP) ----
    scaler.load_state_dict(checkpoint['scaler_state_dict'])

    # ---- LOAD TRAINING STATE ----
    start_epoch = checkpoint['epoch'] + 1
    best_val_acc = checkpoint['best_val_acc']

    print(f"Resuming from epoch {start_epoch} | Best Val Acc: {best_val_acc:.4f}")

else:
    print("No checkpoint found. Starting fresh.")

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Checkpoint found. Loading...
Resuming from epoch 3 | Best Val Acc: 0.9819


In [18]:
class_counts = Counter(train_dataset.targets)
total = sum(class_counts.values())

weights = torch.tensor([
    total / class_counts[0],
    total / class_counts[1]
], dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)

optimizer = optim.AdamW(model.parameters(), lr=5e-5)

In [19]:
from torch import amp

scaler = amp.GradScaler("cuda")

In [20]:
num_epochs = 15
best_val_acc = 0

os.makedirs("checkpoints", exist_ok=True)

for epoch in range(num_epochs):

    print(f"\nEpoch {epoch+1}/{num_epochs}")

    # ---------------- TRAIN ----------------
    model.train()
    train_loss = 0
    train_preds = []
    train_labels = []

    train_bar = tqdm(train_loader, dynamic_ncols=True)

    for images, labels in train_bar:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with amp.autocast("cuda"):# mixed precision
            outputs = model(pixel_values=images)
            loss = criterion(outputs.logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

        preds = torch.argmax(outputs.logits, dim=1)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels.extend(labels.detach().cpu().numpy())

        train_bar.set_postfix(loss=f"{loss.item():.4f}")

    train_acc = accuracy_score(train_labels, train_preds)
    train_loss /= len(train_loader)

    # ---------------- VALIDATION ----------------
    model.eval()
    val_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for images, labels in val_loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with amp.autocast("cuda"):
                outputs = model(pixel_values=images)
                loss = criterion(outputs.logits, labels)

            val_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1)
            val_preds.extend(preds.detach().cpu().numpy())
            val_labels.extend(labels.detach().cpu().numpy())

    val_acc = accuracy_score(val_labels, val_preds)
    val_loss /= len(val_loader)

    # ---------------- SAVE BEST MODEL ----------------
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),  # if using AMP
            'best_val_acc': best_val_acc
        }, "checkpoints/best_model.pth")
        print("Best model saved!")

    print(
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

    torch.cuda.empty_cache()


Epoch 1/15


  0%|                                                                                          | 0/8751 [00:00…

Best model saved!
Train Loss: 0.0211 | Train Acc: 0.9918 | Val Loss: 0.0687 | Val Acc: 0.9805

Epoch 2/15


  0%|                                                                                          | 0/8751 [00:00…

Best model saved!
Train Loss: 0.0185 | Train Acc: 0.9929 | Val Loss: 0.0548 | Val Acc: 0.9818

Epoch 3/15


...

Train Loss: 0.0095 | Train Acc: 0.9967 | Val Loss: 0.0911 | Val Acc: 0.9767

Epoch 14/15


  0%|                                                                                          | 0/8751 [00:00…

Train Loss: 0.0088 | Train Acc: 0.9969 | Val Loss: 0.1199 | Val Acc: 0.9733

Epoch 15/15


  0%|                                                                                          | 0/8751 [00:00…

Train Loss: 0.0091 | Train Acc: 0.9969 | Val Loss: 0.0758 | Val Acc: 0.9777
